# 003 · Advanced Types

> **Syntax › Type Hints** · Python 3.14

Beyond `int`, `list[str]` and generics, the `typing` module provides a vocabulary for describing real-world APIs precisely: aliases, unions, literal values, constants, dictionary shapes, callables, distinct ID types, and type-narrowing functions. This notebook is a guided tour, each with its closest C# counterpart.

**In this notebook you will learn:**

- `type` alias statements (3.12+) vs C#'s `using X = ...`
- `Union` / `Optional` / `|`, `Literal`, and `Final`
- `TypedDict` (with `NotRequired`) for dictionary shapes
- `Callable[[int], str]` vs `Func<int, string>`, and `Any` vs `object`
- `NewType` for strongly-typed IDs, `Self` for fluent methods
- `TypeGuard` / `TypeIs`, `cast()`, and `@overload` vs C# overloads

## C# mental model

| C# | Python |
|---|---|
| `using Point = (int X, int Y);` (C# 12) | `type Point = tuple[int, int]` |
| `int?` / `string?` | `int \| None` (`Optional[int]`) |
| enum / string constants | `Literal["GET", "POST"]` |
| `const` / `readonly` | `Final` |
| anonymous / record type for JSON | `TypedDict` |
| `Func<int, string>`, `Action<int>` | `Callable[[int], str]`, `Callable[[int], None]` |
| `dynamic` | `Any` |
| `object` | `object` |
| `readonly record struct UserId(int Value)` | `UserId = NewType("UserId", int)` |
| `[NotNullWhen(true)]` / `is` patterns | `TypeGuard` / `TypeIs` |
| `(T)x` cast | `cast(T, x)` (no runtime check!) |
| method overloads | `@overload` signatures + one implementation |

As always, all of this is checked by mypy/pyright, not the interpreter *(see Type Hints › 001 · Type Annotations)*.

## Type aliases: the `type` statement

```csharp
// C# 12 — alias any type
using Point = (int X, int Y);
using Inventory = System.Collections.Generic.Dictionary<string, int>;
```

Python 3.12 added a dedicated `type` statement. Aliases can be generic and even recursive, and the right-hand side is evaluated lazily.

In [ ]:
type Point = tuple[float, float]
type Inventory = dict[str, int]
type Pair[T] = tuple[T, T]                                 # generic alias
type Json = dict[str, Json] | list[Json] | str | int | float | bool | None   # recursive!


def distance(a: Point, b: Point) -> float:
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5


print(distance((0, 0), (3, 4)))
print(Point, "->", Point.__value__)
print(type(Point).__name__)

💡 Before 3.12 you'd see plain assignments (`Point = tuple[float, float]`) or `Point: TypeAlias = ...`. They still work; the `type` statement is clearer and supports forward/recursive references.

## Unions and `Optional`

`A | B` means "either an `A` or a `B`". `Optional[X]` is an older spelling of `X | None` — it does **not** mean "optional parameter".

| Modern | Older (still common) |
|---|---|
| `int \| str` | `Union[int, str]` |
| `int \| None` | `Optional[int]` |

In [ ]:
from typing import Optional, Union


def format_id(value: int | str) -> str:
    match value:                       # narrowing by type, like C# switch patterns
        case int():
            return f"#{value:05d}"
        case str():
            return value.upper()


def old_style(x: Union[int, str], y: Optional[int] = None) -> str:   # same meaning
    return f"{x}/{y}"


print(format_id(42), format_id("ab-7"))
print(old_style(1), Optional[int] == (int | None))

## `Literal` — a fixed set of values

`Literal` restricts a value to specific constants. It's a lightweight alternative to an enum *(see Classes & OOP › 009 · Enums)*, very common for string options.

```csharp
// C# would typically use an enum
enum HttpMethod { Get, Post }
```

In [ ]:
from typing import Literal

type HttpMethod = Literal["GET", "POST", "DELETE"]


def request(method: HttpMethod, url: str) -> str:
    return f"{method} {url}"


print(request("GET", "/products"))
print(request("PATCH", "/products"))   # runs! but the checker flags "PATCH"

## `Final` — constants

Python has no `const`. `Final` tells the checker that a name must not be reassigned (≈ `const` / `readonly`). Runtime does not care.

In [ ]:
from typing import Final

MAX_RETRIES: Final = 3          # type inferred as int
API_VERSION: Final[str] = "v2"

MAX_RETRIES = 4                 # checker error: cannot assign to final name
print(MAX_RETRIES)              # ...but the interpreter allows it

## `TypedDict` — describing dictionary shapes

JSON-like data often travels as plain dicts. `TypedDict` describes which keys exist and their value types — the checker then validates key access. At runtime it is a normal `dict`.

```csharp
// C# — you'd usually define a record and deserialize into it
record ProductDto(string Name, decimal Price, string[]? Tags = null);
```

In [ ]:
from typing import NotRequired, TypedDict


class ProductDto(TypedDict):
    name: str
    price: float
    tags: NotRequired[list[str]]     # key may be missing


laptop: ProductDto = {"name": "laptop", "price": 999.0}
mouse: ProductDto = {"name": "mouse", "price": 19.5, "tags": ["usb"]}

for p in (laptop, mouse):
    print(p["name"], p.get("tags", []))

print(type(laptop))                          # just a dict
print(ProductDto.__required_keys__, ProductDto.__optional_keys__)

💡 Use `total=False` on the class to make every key optional, and `Required[...]` to opt individual keys back in. For data with behaviour, prefer a `dataclass` *(see Classes & OOP › 008 · Dataclasses)*.

## `Callable` vs `Func` / `Action`

```csharp
// C#
Func<int, string> format = n => $"#{n}";
Action<string> log = Console.WriteLine;
Func<int, int, int> add = (a, b) => a + b;
```

| C# | Python |
|---|---|
| `Func<int, string>` | `Callable[[int], str]` |
| `Func<int, int, int>` | `Callable[[int, int], int]` |
| `Action<string>` | `Callable[[str], None]` |
| `Func<string>` | `Callable[[], str]` |
| any signature | `Callable[..., Any]` |

In [ ]:
from collections.abc import Callable


def apply_all(values: list[int], f: Callable[[int], str]) -> list[str]:
    return [f(v) for v in values]


def notify(handler: Callable[[str], None]) -> None:
    handler("saved")


print(apply_all([1, 2], lambda n: f"#{n}"))
notify(print)

## `Any` vs `object`

Both accept any value, but they mean opposite things to the checker:

- `object` — "could be anything, so you may only do what **every** object supports" (safe, like C# `object`)
- `Any` — "turn type checking **off** for this value" (like C# `dynamic`)

In [ ]:
from typing import Any


def with_object(x: object) -> None:
    print(len(str(x)))     # fine
    # x.upper()            # checker error: object has no attribute "upper"


def with_any(x: Any) -> None:
    print(x.upper())       # checker allows anything — fails only at runtime if wrong


with_object(123)
with_any("ok")
try:
    with_any(123)
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

💡 Prefer `object` when you just need "anything". Reach for `Any` only at boundaries with untyped code.

## `NewType` — distinct ID types

Mixing up a `user_id` and an `order_id` is a classic bug. In C# you'd wrap them in tiny record structs. `NewType` creates a distinct type **for the checker** with zero runtime cost — at runtime it's just the underlying value.

```csharp
// C#
readonly record struct UserId(int Value);
readonly record struct OrderId(int Value);

Order? FindOrder(OrderId id) => ...;
FindOrder(new UserId(7));   // compile error
```

In [ ]:
from typing import NewType

UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)


def find_order(order_id: OrderId) -> str:
    return f"order {order_id}"


uid = UserId(7)
oid = OrderId(1001)
print(find_order(oid))
print(find_order(uid))     # checker error: UserId is not OrderId — runtime doesn't mind
print(type(uid), uid + 1)  # it's a plain int at runtime

## `Self` — methods returning their own type

Fluent APIs return `self`. Annotating with the class name breaks for subclasses; `Self` (3.11+) means "the type of the actual instance". C# has no direct equivalent — you'd use the "curiously recurring" pattern `Builder<TSelf> where TSelf : Builder<TSelf>`.

In [ ]:
from typing import Self


class QueryBuilder:
    def __init__(self) -> None:
        self.parts: list[str] = []

    def where(self, cond: str) -> Self:
        self.parts.append(f"WHERE {cond}")
        return self


class PagedQueryBuilder(QueryBuilder):
    def limit(self, n: int) -> Self:
        self.parts.append(f"LIMIT {n}")
        return self


q = PagedQueryBuilder().where("price > 10").limit(5)   # .where() returns PagedQueryBuilder
print(" ".join(q.parts))

## `TypeGuard` and `TypeIs` — custom narrowing

The checker narrows types after `isinstance` checks. To make your **own** helper function narrow a type, annotate its return with `TypeIs[T]` (3.13+) or `TypeGuard[T]` (3.10+). It's similar to C#'s `[NotNullWhen(true)]` attribute or a method with an `out` parameter used in `if (TryParse(...))`.

- `TypeIs[T]` — narrows in **both** branches (`True` → `T`, `False` → "not `T`"). Prefer it.
- `TypeGuard[T]` — narrows only in the `True` branch, and can narrow to a type that isn't a subtype of the input (e.g. `list[object]` → `list[str]`).

In [ ]:
from typing import TypeGuard, TypeIs


def is_str(value: object) -> TypeIs[str]:
    return isinstance(value, str)


def all_strings(values: list[object]) -> TypeGuard[list[str]]:
    return all(isinstance(v, str) for v in values)


def shout(value: int | str) -> str:
    if is_str(value):
        return value.upper()     # checker: value is str
    return str(value * 2)        # checker: value is int (TypeIs narrows the else branch too)


print(shout("hi"), shout(21))
items: list[object] = ["a", "b"]
if all_strings(items):
    print(", ".join(items))      # checker: items is list[str]

⚠️ **Gotcha:** only use `TypeIs[T]` when the function returns `True` **exactly** when the value is a `T`. A check like "is a *positive* int" returns `False` for `-1`, which is still an `int` — use `TypeGuard[int]` there, so the checker doesn't wrongly narrow the `else` branch to "not int".

## `cast()` — trust me, checker

`cast(T, value)` tells the checker to treat `value` as `T`. Unlike a C# cast `(T)x`, it performs **no runtime check** and never throws — it just returns the value unchanged.

In [ ]:
from typing import cast

raw: object = {"name": "alice"}
user = cast(dict[str, str], raw)     # checker now treats user as dict[str, str]
print(user["name"])

lie = cast(int, "definitely not an int")   # no error at runtime!
print(lie, type(lie).__name__)

⚠️ **Gotcha:** `cast` is an escape hatch, not a conversion. To *convert*, call the type (`int("42")`); to *check*, use `isinstance`.

## `@overload` vs C# overloads

C# picks between multiple method bodies at compile time. Python has only **one** function per name — defining it twice simply replaces the first:

In [ ]:
def area(side):
    return side * side


def area(width, height):    # replaces the previous definition entirely
    return width * height


try:
    area(3)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

`@overload` lets you declare several **signatures** for the type checker, followed by a single real implementation that handles all cases:

```csharp
// C#
static int Parse(string s) => int.Parse(s);
static int[] Parse(string[] items) => items.Select(int.Parse).ToArray();
```

In [ ]:
from typing import overload


@overload
def parse(data: str) -> int: ...
@overload
def parse(data: list[str]) -> list[int]: ...


def parse(data: str | list[str]) -> int | list[int]:   # the one real implementation
    if isinstance(data, str):
        return int(data)
    return [int(x) for x in data]


print(parse("42") + 1)          # checker knows: int
print(parse(["1", "2"]))        # checker knows: list[int]

💡 Often default arguments or a union parameter are simpler than overloads. Use `@overload` when the **return type depends on the argument type**.

## Summary

| Concept | C# | Python |
|---|---|---|
| Type alias | `using Point = (int, int);` | `type Point = tuple[int, int]` |
| Union | — (inheritance / `OneOf`) | `int \| str` |
| Nullable | `int?` | `int \| None` / `Optional[int]` |
| Fixed values | `enum` | `Literal["a", "b"]` |
| Constant | `const` / `readonly` | `Final` |
| Dict shape | DTO record | `TypedDict` + `NotRequired` |
| Function type | `Func<int, string>` | `Callable[[int], str]` |
| Escape hatch | `dynamic` | `Any` |
| Strong ID | `record struct UserId(int)` | `NewType("UserId", int)` |
| Own type | CRTP `TSelf` | `Self` |
| Custom narrowing | `[NotNullWhen(true)]` | `TypeIs[T]` / `TypeGuard[T]` |
| Cast | `(T)x` — checked | `cast(T, x)` — unchecked |
| Overloads | multiple bodies | `@overload` stubs + one body |

## Exercises

1. Define `type Matrix = list[list[float]]` and write `def transpose(m: Matrix) -> Matrix`.
2. Create a `TypedDict` called `OrderDto` with `id: int`, `total: float` and an optional `coupon: str`. Write a function that returns the coupon or `"none"`.
3. Write a narrowing function `is_positive_int(x: object)` and use it to filter `[3, "a", -1, 7, None]` down to the positive ints. Should it return `TypeIs[int]` or `TypeGuard[int]`?
4. Define `ProductId = NewType("ProductId", str)`. What does `type(ProductId("p-1"))` return, and why?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from typing import NewType, NotRequired, TypedDict, TypeGuard

# 1
type Matrix = list[list[float]]

def transpose(m: Matrix) -> Matrix:
    return [list(row) for row in zip(*m)]

print(transpose([[1, 2, 3], [4, 5, 6]]))   # [[1, 4], [2, 5], [3, 6]]

# 2
class OrderDto(TypedDict):
    id: int
    total: float
    coupon: NotRequired[str]

def coupon_of(order: OrderDto) -> str:
    return order.get("coupon", "none")

print(coupon_of({"id": 1, "total": 9.5}), coupon_of({"id": 2, "total": 5.0, "coupon": "SAVE5"}))

# 3 — TypeGuard, not TypeIs: a False result (e.g. for -1) does NOT mean
#     "x is not an int", so the checker must not narrow the else branch.
#     (bool is a subclass of int; exclude it if you want "real" ints only.)
def is_positive_int(x: object) -> TypeGuard[int]:
    return isinstance(x, int) and not isinstance(x, bool) and x > 0

print([x for x in [3, "a", -1, 7, None] if is_positive_int(x)])   # [3, 7]

# 4 — <class 'str'>: NewType only exists for the type checker;
#     at runtime ProductId("p-1") just returns the same str.
ProductId = NewType("ProductId", str)
print(type(ProductId("p-1")))
```

</details>